In [1]:
# ============================================================
# FUNCTION 6 — WEEK 6 CLEAN REBUILD
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel as C,
    Matern,
    WhiteKernel
)

# ------------------------------------------------------------
# 1. Load original Function 6 data
# ------------------------------------------------------------

X = np.load("function6/initial_inputs.npy")
Y = np.load("function6/initial_outputs.npy").reshape(-1)

# ------------------------------------------------------------
# 2. Add Weeks 1–5 exactly once
# ------------------------------------------------------------

weekly_X = np.array([
    [0.296485, 0.322087, 0.332446, 0.760368, 0.040944],  # W1
    [0.351404, 0.334902, 0.511830, 0.859800, 0.149390],  # W2
    [0.469953, 0.133458, 0.475744, 0.973553, 0.000000],  # W3
    [0.442929, 0.409333, 0.631830, 0.739800, 0.129381],  # W4
    [0.460592, 0.370370, 0.863372, 0.658016, 0.095986]   # W5
])

weekly_Y = np.array([
    -0.5070321901597052,
    -0.39798887850658554,
    -0.862627252477251,
    -0.19517557780237038,
    -0.37349175750659697
])

X = np.vstack([
    X,
    weekly_X
])

Y = np.concatenate([
    Y,
    weekly_Y
])

best_index = np.argmax(Y)
best_x = X[best_index]
best_y = Y[best_index]

print("Week 6 X shape:", X.shape)
print("Week 6 Y shape:", Y.shape)

print("\nBest observed point:")
print(best_x)

print("Best observed output:")
print(best_y)

print("\nWeek 5 point:")
print(weekly_X[-1])

print("Week 5 output:")
print(weekly_Y[-1])

Week 6 X shape: (25, 5)
Week 6 Y shape: (25,)

Best observed point:
[0.442929 0.409333 0.63183  0.7398   0.129381]
Best observed output:
-0.19517557780237038

Week 5 point:
[0.460592 0.37037  0.863372 0.658016 0.095986]
Week 5 output:
-0.37349175750659697


In [2]:
# ============================================================
# 3. FIT ARD GAUSSIAN PROCESS
# ============================================================

kernel = (
    C(
        1.0,
        (1e-3, 1e3)
    )
    *
    Matern(
        length_scale=[
            0.40,
            0.40,
            0.35,
            0.40,
            0.35
        ],
        length_scale_bounds=(
            0.01,
            3.0
        ),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(
            1e-10,
            1e-2
        )
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=66
)

gp.fit(
    X,
    Y
)

print("Fitted kernel:")
print(gp.kernel_)

lengthscales = np.asarray(
    gp.kernel_.k1.k2.length_scale
)

importance = 1 / lengthscales
importance = importance / importance.sum()

print("\nFitted lengthscales:")
print(lengthscales)

print("\nNormalised input influence:")

for i, value in enumerate(
    importance,
    start=1
):
    print(f"x{i}: {value:.4f}")

Fitted kernel:
1.19**2 * Matern(length_scale=[0.669, 0.804, 1.09, 0.801, 0.882], nu=2.5) + WhiteKernel(noise_level=1.84e-10)

Fitted lengthscales:
[0.66894322 0.80420073 1.08838347 0.80062836 0.88248958]

Normalised input influence:
x1: 0.2475
x2: 0.2059
x3: 0.1521
x4: 0.2068
x5: 0.1876


In [3]:
# ============================================================
# 4. WEEK 6 CANDIDATE GENERATION
#
# Main focus returns to Week 4.
# x3 receives smaller movements because Week 5 moved too far.
# ============================================================

rng = np.random.default_rng(66)

week4_point = np.array([
    0.442929,
    0.409333,
    0.631830,
    0.739800,
    0.129381
])

week2_point = np.array([
    0.351404,
    0.334902,
    0.511830,
    0.859800,
    0.149390
])

week5_point = np.array([
    0.460592,
    0.370370,
    0.863372,
    0.658016,
    0.095986
])

# ------------------------------------------------------------
# A. Very tight search around Week 4
# ------------------------------------------------------------

very_local = rng.normal(
    loc=week4_point,
    scale=[
        0.010,
        0.010,
        0.010,
        0.010,
        0.008
    ],
    size=(20_000, 5)
)

# ------------------------------------------------------------
# B. Local search around Week 4
# ------------------------------------------------------------

local = rng.normal(
    loc=week4_point,
    scale=[
        0.025,
        0.025,
        0.025,
        0.025,
        0.020
    ],
    size=(20_000, 5)
)

# ------------------------------------------------------------
# C. Slightly wider local search
#
# Still much smaller x3 movement than Week 5.
# ------------------------------------------------------------

wider_local = rng.normal(
    loc=week4_point,
    scale=[
        0.050,
        0.050,
        0.045,
        0.050,
        0.040
    ],
    size=(10_000, 5)
)

# ------------------------------------------------------------
# D. Blend between Week 2 and Week 4
# ------------------------------------------------------------

weights = rng.uniform(
    0,
    1,
    size=(10_000, 1)
)

blended = (
    weights * week4_point
    +
    (1 - weights) * week2_point
)

blended += rng.normal(
    0,
    [
        0.015,
        0.015,
        0.015,
        0.015,
        0.012
    ],
    size=(10_000, 5)
)

# ------------------------------------------------------------
# E. Controlled search slightly around x3
#
# Keep x3 close to the successful Week 4 value.
# ------------------------------------------------------------

controlled_x3 = np.column_stack([
    rng.normal(0.442929, 0.025, 8_000),
    rng.normal(0.409333, 0.025, 8_000),
    rng.uniform(0.56, 0.70, 8_000),
    rng.normal(0.739800, 0.030, 8_000),
    rng.normal(0.129381, 0.020, 8_000)
])

candidates = np.vstack([
    very_local,
    local,
    wider_local,
    blended,
    controlled_x3
])

candidates = np.clip(
    candidates,
    0,
    1
)

print(
    "Candidates before filtering:",
    len(candidates)
)

Candidates before filtering: 68000


In [4]:
# ============================================================
# 5. REMOVE NEAR-DUPLICATES
# ============================================================

tree = cKDTree(X)

minimum_distance, _ = tree.query(
    candidates,
    k=1
)

keep = (
    minimum_distance > 0.0035
)

candidates = candidates[
    keep
]

print(
    "Candidates after filtering:",
    len(candidates)
)

Candidates after filtering: 67993


In [5]:
# ============================================================
# 6. GP PREDICTIONS
# ============================================================

mean, std = gp.predict(
    candidates,
    return_std=True
)

# ------------------------------------------------------------
# Expected Improvement
# ------------------------------------------------------------

xi = 0.001

improvement = (
    mean
    - best_y
    - xi
)

with np.errstate(
    divide="ignore",
    invalid="ignore"
):

    z = improvement / std

    ei = (
        improvement * norm.cdf(z)
        +
        std * norm.pdf(z)
    )

ei[std < 1e-12] = 0

# ------------------------------------------------------------
# Small UCB component
# ------------------------------------------------------------

kappa = 0.30

ucb = (
    mean
    + kappa * std
)

In [6]:
# ============================================================
# 7. FILTER WEAK PREDICTED CANDIDATES
# ============================================================

mean_filter = (
    mean >= best_y - 0.08
)

if np.sum(mean_filter) < 100:
    mean_filter = (
        mean >= np.percentile(
            mean,
            90
        )
    )

filtered_candidates = candidates[
    mean_filter
]

filtered_mean = mean[
    mean_filter
]

filtered_std = std[
    mean_filter
]

filtered_ei = ei[
    mean_filter
]

filtered_ucb = ucb[
    mean_filter
]

print(
    "Candidates passing mean filter:",
    len(filtered_candidates)
)

Candidates passing mean filter: 61898


In [7]:
# ============================================================
# 8. ACQUISITION SCORE
# ============================================================

ei_norm = (
    filtered_ei
    - filtered_ei.min()
) / (
    np.ptp(filtered_ei)
    + 1e-12
)

ucb_norm = (
    filtered_ucb
    - filtered_ucb.min()
) / (
    np.ptp(filtered_ucb)
    + 1e-12
)

acquisition = (
    0.90 * ei_norm
    +
    0.10 * ucb_norm
)

chosen_index = np.argmax(
    acquisition
)

week6_query = filtered_candidates[
    chosen_index
]

In [8]:
# ============================================================
# 9. FUNCTION 6 — WEEK 6 PORTAL OUTPUT
# ============================================================

print("\nSuggested Week 6 query:")
print(week6_query)

print("\nPortal format:")
print(
    "-".join(
        f"{value:.6f}"
        for value in week6_query
    )
)

print(
    "\nPredicted mean:",
    filtered_mean[chosen_index]
)

print(
    "Predicted uncertainty:",
    filtered_std[chosen_index]
)

print(
    "Expected Improvement:",
    filtered_ei[chosen_index]
)

print(
    "UCB:",
    filtered_ucb[chosen_index]
)

print(
    "Distance from Week 4 best:",
    np.linalg.norm(
        week6_query
        - week4_point
    )
)

print(
    "Distance from Week 5:",
    np.linalg.norm(
        week6_query
        - week5_point
    )
)

print(
    "x3 change from Week 4:",
    week6_query[2]
    - week4_point[2]
)


Suggested Week 6 query:
[0.44604417 0.45405752 0.560136   0.68430965 0.0493892 ]

Portal format:
0.446044-0.454058-0.560136-0.684310-0.049389

Predicted mean: -0.19958430406627037
Predicted uncertainty: 0.06301707383643897
Expected Improvement: 0.022528355386190767
UCB: -0.1806791819153387
Distance from Week 4 best: 0.12894915425038603
Distance from Week 5: 0.3194212404052879
x3 change from Week 4: -0.07169399742912286
